### Create the LunarLander environment

In [1]:
import gym
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


env = gym.make("LunarLander-v2")

### Setup R packages

In [2]:
from cxrl.lib.r_to_py import setup_R
setup_R()

[]


### A policy (model-free) for LunarLander

In [3]:
def policy(state):
    """
    Simple heuristic policy for Gym/Gymnasium LunarLander with discrete actions.

    state:
        x, y, vx, vy, angle, angular_velocity, left_contact, right_contact

    returns:
        action in {0, 1, 2, 3}
    """
    x, y, vx, vy, angle, angular_velocity, left_contact, right_contact = state

    # 1. Point the lander toward the center.
    # If x > 0, lander is right of pad, so target a positive angle
    # to move back left. vx helps damp horizontal motion.
    target_angle = 0.5 * x + 1.0 * vx
    target_angle = np.clip(target_angle, -0.4, 0.4)

    # 2. When far from center, stay higher; when near center, descend.
    target_y = 0.55 * abs(x)

    # 3. PD-style control errors.
    angle_control = 0.5 * (target_angle - angle) - 1.0 * angular_velocity
    hover_control = 0.5 * (target_y - y) - 0.5 * vy

    # 4. Once a leg touches, stop trying to rotate; just reduce falling speed.
    if left_contact or right_contact:
        angle_control = 0.0
        hover_control = -0.5 * vy

    # 5. Convert desired controls to discrete actions.
    if hover_control > abs(angle_control) and hover_control > 0.05:
        return 2      # main engine
    elif angle_control < -0.05:
        return 3      # right orientation engine
    elif angle_control > 0.05:
        return 1      # left orientation engine
    else:
        return 0      # do nothing

### Infer the global causal graph from episodes

In [4]:
from cxrl.lib.xrl import XRL

# Names for interpretable representations: 8 state variables + 4 one-hot actions.
state_names = [
    'x', 'y', 'vx', 'vy', 'angle', 'angular_velocity',
    'left_contact', 'right_contact'
]
action_names = [
    'do_nothing', 'fire_left_engine', 'fire_main_engine', 'fire_right_engine'
]
feature_names = state_names + action_names

# LunarLander inference is substantially heavier than CartPole because there are
# more target nodes. Increase NEPISODES for a larger evaluation run.
NEPISODES = 30
xrl = XRL(
    env,
    pi=policy,
    repr_names=feature_names,
    verbose=True,
    one_hot_actions=True,
    repr_integration=False,
    nepisodes=NEPISODES,
)

experience size= 7363
 causes: ['x', 'vx'] -> x
 causes: ['y', 'vx', 'right_contact'] -> y
 causes: ['vx'] -> vx
 causes: ['y', 'vy', 'left_contact', 'do_nothing', 'fire_left_engine'] -> vy
 causes: ['angle', 'angular_velocity'] -> angle
 causes: ['angular_velocity', 'left_contact', 'right_contact'] -> angular_velocity
 causes: ['y', 'left_contact', 'right_contact'] -> left_contact
 causes: ['y', 'left_contact', 'right_contact'] -> right_contact
 causes: ['x', 'vy', 'left_contact', 'right_contact', 'do_nothing', 'fire_main_engine'] -> do_nothing
 causes: ['vx', 'angle', 'angular_velocity', 'fire_left_engine', 'fire_main_engine'] -> fire_left_engine
 causes: ['y', 'vy', 'left_contact', 'right_contact', 'fire_main_engine'] -> fire_main_engine
 causes: ['vx', 'angle', 'angular_velocity', 'left_contact', 'right_contact', 'fire_main_engine', 'fire_right_engine'] -> fire_right_engine
BART SLA elapsed time: 0.0 seconds


In [5]:
states, actions, rewards, states_new = xrl.replay_buffer

from cxrl.lib.utils import convert_and_expand
states = convert_and_expand(states).astype(float)
actions = convert_and_expand(actions).astype(int)

# Build a fixed 4-column one-hot action matrix so replay columns always match
# feature_names, even if a short run happens not to sample every action.
action_one_hot = np.zeros((actions.shape[0], len(action_names)), dtype=float)
action_one_hot[np.arange(actions.shape[0]), actions.squeeze()] = 1.0

replay = pd.DataFrame(
    np.concatenate((states, action_one_hot), axis=1),
    columns=feature_names,
)
replay.head()

,x,y,vx,vy,angle,angular_velocity,left_contact,right_contact,do_nothing,fire_left_engine,fire_main_engine,fire_right_engine
0,0.002194,1.417407,0.222225,0.288281,-0.002536,-0.050337,0.0,0.0,0.0,1.0,0.0,0.0
1,0.004292,1.423314,0.209794,0.262537,-0.002590,-0.001088,0.0,0.0,0.0,1.0,0.0,0.0
2,0.006302,1.428633,0.198896,0.236407,-0.000459,0.042633,0.0,0.0,0.0,1.0,0.0,0.0
3,0.008240,1.433342,0.189829,0.209268,0.003489,0.078959,0.0,0.0,1.0,0.0,0.0,0.0
4,0.010178,1.437450,0.189817,0.182599,0.007436,0.078940,0.0,0.0,1.0,0.0,0.0,0.0


### Evaluation

Extract timesteps where at least one action is a target node in the local causal model

In [6]:
action_indices = list(range(len(state_names), len(feature_names)))
valid_steps = []

for step in range(replay.shape[0] - 1):
    query = replay.iloc[step:step + 1]
    for action_index in action_indices:
        local_causes = xrl.scm[action_index].local_treatments(query)
        n_local_causes = sum(
            1 for cause in np.nonzero(local_causes[0])[0]
            if cause != action_index
        )
        if n_local_causes > 1:
            valid_steps.append(step)
            break

if not valid_steps:
    print('No action-node local models with more than one cause were found; using all non-final replay steps.')
    valid_steps = list(range(replay.shape[0] - 1))

print('number of steps for evaluations:', len(valid_steps))

number of steps for evaluations: 7362


Create representation pools for timestep t and t+1

In [7]:
valid_steps_tp1 = (np.array(valid_steps) + 1).tolist()
eval_replay_t = replay.iloc[valid_steps, :]
eval_replay_tp1 = replay.iloc[valid_steps_tp1, :]
print(eval_replay_t.head())

eval_replay_t = eval_replay_t.to_numpy()
eval_replay_tp1 = eval_replay_tp1.to_numpy()

          x         y        vx        vy     angle  angular_velocity  \
0  0.002194  1.417407  0.222225  0.288281 -0.002536         -0.050337   
1  0.004292  1.423314  0.209794  0.262537 -0.002590         -0.001088   
2  0.006302  1.428633  0.198896  0.236407 -0.000459          0.042633   
3  0.008240  1.433342  0.189829  0.209268  0.003489          0.078959   
4  0.010178  1.437450  0.189817  0.182599  0.007436          0.078940   

   left_contact  right_contact  do_nothing  fire_left_engine  \
0           0.0            0.0         0.0               1.0   
1           0.0            0.0         0.0               1.0   
2           0.0            0.0         0.0               1.0   
3           0.0            0.0         1.0               0.0   
4           0.0            0.0         1.0               0.0   

   fire_main_engine  fire_right_engine  
0               0.0                0.0  
1               0.0                0.0  
2               0.0                0.0  
3           

Create holdout dataset

In [8]:
features_t = eval_replay_t
action_tp1_one_hot = eval_replay_tp1[:, action_indices]
action_tp1 = action_tp1_one_hot.argmax(axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    features_t,
    action_tp1,
    test_size=0.4,
    random_state=10,
    stratify=action_tp1 if len(np.unique(action_tp1)) > 1 else None,
)
train_idx, test_idx = train_test_split(
    np.arange(len(valid_steps)),
    test_size=0.4,
    random_state=10,
    stratify=action_tp1 if len(np.unique(action_tp1)) > 1 else None,
)

y_train_one_hot = np.eye(len(action_names))[y_train]
y_test_one_hot = np.eye(len(action_names))[y_test]

print('train action counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('test action counts:', dict(zip(*np.unique(y_test, return_counts=True))))

train action counts: {0: 2320, 1: 217, 2: 1663, 3: 217}
test action counts: {0: 1547, 1: 144, 2: 1109, 3: 145}


Decision Tree accuracy

In [9]:
clf_tree = DecisionTreeClassifier(random_state=1)
clf_tree.fit(X_train, y_train)
y_pred_tree = clf_tree.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)
print(f'Decision Tree Accuracy: {accuracy_tree * 100:.2f}%')

Decision Tree Accuracy: 81.09%


Logistic Regression accuracy

In [10]:
clf_lr = LogisticRegression(random_state=1, max_iter=1000).fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression Accuracy: {accuracy_lr * 100:.2f}%')

Logistic Regression Accuracy: 80.24%


Build BART inference models for action targets

In [11]:
from cxrl.lib.models import BARTRegressor

def feature_index_from_causal_node(node):
    if node.startswith('state_') and node.endswith('_t'):
        name = node[len('state_'):-len('_t')]
    elif node.startswith('action_') and node.endswith('_t'):
        name = node[len('action_'):-len('_t')]
    else:
        return None

    try:
        return feature_names.index(name)
    except ValueError:
        return None

all_cols = list(range(len(feature_names)))
num_cols = [0, 1, 2, 3, 4, 5]
action_global_causes = {}
hash_y_bart = {}

for action_index, action_name in zip(action_indices, action_names):
    target_node = f'action_{action_name}_t'
    causes = sorted(
        idx for idx in (
            feature_index_from_causal_node(pred)
            for pred in xrl.causalG.predecessors(target_node)
        )
        if idx is not None and idx != action_index
    )
    action_global_causes[action_index] = causes
    print(action_name, '<-', [feature_names[i] for i in causes])

    covars = [i for i in all_cols if i not in causes]
    y_train_action = y_train_one_hot[:, action_index - len(state_names)]
    for treat in causes:
        treat_and_covars = [treat] + covars
        bart = BARTRegressor()
        bart.fit(X_train[:, treat_and_covars], y_train_action)
        hash_y_bart[(action_index, treat)] = bart.predict(X_test[:, treat_and_covars])

do_nothing <- ['x', 'vy', 'left_contact', 'right_contact', 'fire_main_engine']
fire_left_engine <- ['vx', 'angle', 'angular_velocity', 'fire_main_engine']
fire_main_engine <- ['y', 'vy', 'left_contact', 'right_contact']
fire_right_engine <- ['vx', 'angle', 'angular_velocity', 'left_contact', 'right_contact', 'fire_main_engine']


Predict next action using local causal models weighted by edge strengths

In [12]:
action_priors = y_train_one_hot.mean(axis=0)
y_pred_scores = np.zeros((y_test.shape[0], len(action_names)))

for row_index, step in enumerate(test_idx):
    query = replay.iloc[valid_steps[step]:valid_steps[step] + 1]

    for action_offset, action_index in enumerate(action_indices):
        local_causes = xrl.scm[action_index].local_treatments(query)
        total_weight = 0.0
        y_bart_pred = 0.0

        for treat in np.nonzero(local_causes[0])[0]:
            if (action_index, treat) not in hash_y_bart:
                continue

            weight = abs(
                xrl.scm[action_index]
                .ITE(query, int(treat), additive=(treat in num_cols))
                .mean()
            )
            total_weight += weight
            y_bart_pred += hash_y_bart[(action_index, treat)][row_index] * weight

        if total_weight > 0:
            y_pred_scores[row_index, action_offset] = y_bart_pred / total_weight
        else:
            y_pred_scores[row_index, action_offset] = action_priors[action_offset]

y_pred_xrl = y_pred_scores.argmax(axis=1)

Our model accuracy

In [13]:
accuracy_bart = accuracy_score(y_test, y_pred_xrl)
print(f'BART/XRL Accuracy: {accuracy_bart * 100:.2f}%')

# results = pd.DataFrame({
#     'actual': [action_names[i] for i in y_test],
#     'decision_tree': [action_names[i] for i in y_pred_tree],
#     'logistic_regression': [action_names[i] for i in y_pred_lr],
#     'bart_xrl': [action_names[i] for i in y_pred_xrl],
# })
# results.head(10)

BART/XRL Accuracy: 84.28%
